In [1]:
print("Hello World")

Hello World


In [3]:
import re
import sys
from pathlib import Path

from langchain_community.document_loaders import WebBaseLoader

cwd = Path.cwd()
for candidate in (cwd, cwd.parent):
    if (candidate / "helpers" / "common.py").exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))
        break

from helpers.common import together_ai_llm, langfuse, langfuse_handler


In [10]:
urls = [
  "https://pulse.zerodha.com/",
  "https://groww.in/market-news/stocks"
]

loader = WebBaseLoader(urls)


In [11]:
pages = []
for doc in loader.lazy_load():
    pages.append(doc)



In [12]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])

def text_clean(text):
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\t+', '\t', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [13]:
context = format_docs(pages)
cleaned_context = text_clean(context)
len(cleaned_context)

113903

In [17]:
# Chunk the data 20_000 words
def chunk_text(text, chunk_size, overlap=100):
  chunks = []
  for i in range(0, len(text), chunk_size - overlap):
    chunk = text[i: i + chunk_size]
    chunks.append(chunk)
  return chunks

chunks = chunk_text(cleaned_context, 20_000)


In [7]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

system_message = """
You are helpful AI assistant who answer user question based on the provided context.
"""
human_message = """
Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don't know".
### Context:
{context}

### Question:
{question}

### Answer:
"""

system_prompt = SystemMessagePromptTemplate.from_template(system_message)
human_prompt = HumanMessagePromptTemplate.from_template(human_message)

messages = [system_prompt, human_prompt]
chat_prompt = ChatPromptTemplate.from_messages(messages)
# chat_prompt.invoke({
#   'context': "Hello",
#   "question":"You Smart?"
# })

# question_1 = "Extract stock market news from the given text."
# question_2 = "Write a detailed market news report in markdown format. Think carefully then write the report."


In [19]:
chain = chat_prompt | together_ai_llm | StrOutputParser()
chunk_summaries = []

for chunk in chunks:
  response = chain.invoke({
    "context": chunk,
    "question": "Extract stock market news from the given text."
  },
  config={"callbacks": [langfuse_handler],"run_name": "Summarize Stock Market Chunks",    "metadata": {"langfuse_tags": ["together/gpt-oss"]},},
  )
  chunk_summaries.append(response)
  


In [27]:
import os
os.makedirs("data", exist_ok=True)

for i, chunk_summary in enumerate(chunk_summaries):
  with open(f"data/summary_{i+1}.md", "w", encoding="utf-8") as file:
    file.write(chunk_summary)


In [31]:
# Create a detailed report from those summaries
all_summaries = "\n".join([summary for summary in chunk_summaries])

detailed_report = chain.invoke({
  "context": all_summaries,
  "question": "Write a detailed market news report in markdown format. Think carefully and then write the report."
},
  config={"callbacks": [langfuse_handler],"run_name": "Detailed Market News Report",    "metadata": {"langfuse_tags": ["together/gpt-oss"]},},
)

In [33]:
with open("data/report.md", "w", encoding="utf-8") as file:
  file.write(detailed_report)

In [32]:
langfuse.flush()

In [ ]:
# from pathlib import Path

# path = Path("report.md")
# with open(f"{path}", "w", encoding="utf-8") as file:
#   file.write("## Report Bro")

